# 06 – Chỉ mục FAISS 



In [3]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
sys.path.insert(0, str(PROJECT_ROOT))

CHUNK_DIR = PROJECT_ROOT / "knowledge" / "chunks"
CHUNKS_PATH = CHUNK_DIR / "chunks.jsonl"
INDEX_DIR = PROJECT_ROOT / "knowledge" / "index"
INDEX_PATH = INDEX_DIR / "faiss.index"
META_PATH = INDEX_DIR / "chunks_meta.json"
INFO_PATH = INDEX_DIR / "build_info.json"

YAML_FILES = [
    "chunks_tcmr_va_sang_loc.yaml",
    "chunks_hdsd_11_san_pham.yaml",
    "chunks_who_16_bai.yaml",
]

print("PROJECT_ROOT =", PROJECT_ROOT)
for name in YAML_FILES:
    p = CHUNK_DIR / name
    print(f"  {name}: exists={p.exists()}")


PROJECT_ROOT = d:\TTTN-VaxCare-2026\vaxcare-ai
  chunks_tcmr_va_sang_loc.yaml: exists=True
  chunks_hdsd_11_san_pham.yaml: exists=True
  chunks_who_16_bai.yaml: exists=True


## 1) Gộp file chunk → `chunks.jsonl` (đúng `chunk_id`)




In [4]:
"""Gộp list `- id:` → chunks.jsonl. Parser thủ công, không cần PyYAML."""
import re

def parse_chunk_file(path: Path) -> list[dict]:
    raw = path.read_text(encoding="utf-8")
    parts = re.split(r"(?m)^- id:\s*", raw)
    rows = []
    for part in parts[1:]:
        lines = part.splitlines()
        if not lines:
            continue
        cid = lines[0].strip().strip("\"'")
        fields: dict[str, str] = {"id": cid}
        i = 1
        while i < len(lines):
            line = lines[i]
            m = re.match(r"^  ([a-z_]+):\s*(.*)$", line)
            if not m:
                i += 1
                continue
            key, val = m.group(1), m.group(2)
            multiline = (
                val in ("", "|", ">")
                or (val.startswith("'") and val.count("'") == 1)
                or (val.startswith('"') and val.count('"') == 1)
                or (val.startswith("'") and not val.endswith("'"))
                or (val.startswith('"') and not val.endswith('"'))
            )
            if multiline:
                buf = []
                if val not in ("", "|", ">"):
                    buf.append(val.lstrip("'\""))
                i += 1
                while i < len(lines):
                    nxt = lines[i]
                    if re.match(r"^  [a-z_]+:\s*", nxt):
                        break
                    if nxt.startswith("  "):
                        buf.append(nxt.strip().strip("'\""))
                        i += 1
                    else:
                        break
                fields[key] = " ".join(buf).strip().strip("'\"")
                continue
            val = val.strip()
            if len(val) >= 2 and val[0] == val[-1] and val[0] in "'\"":
                val = val[1:-1]
            fields[key] = val
            i += 1
        if fields.get("id") and fields.get("text"):
            rows.append(fields)
    return rows

rows = []
seen: set[str] = set()
for name in YAML_FILES:
    path = CHUNK_DIR / name
    if not path.exists():
        print("SKIP missing", path)
        continue
    items = parse_chunk_file(path)
    n = 0
    for it in items:
        cid = str(it.get("id") or "").strip()
        text = (it.get("text") or "").strip()
        if not cid or not text:
            continue
        if cid in seen:
            print("DUP", cid)
            continue
        seen.add(cid)
        vid = it.get("vaccine_id")
        vaccine_ids = []
        if vid is not None and str(vid).lower() not in ("null", "none", ""):
            try:
                vaccine_ids = [int(vid)]
            except (TypeError, ValueError):
                pass
        tier = "B" if "who" in cid else "A"
        rows.append({
            "chunk_id": cid,
            "doc_id": cid.rsplit("-", 1)[0] if "-" in cid else cid,
            "title": it.get("topic") or cid,
            "text": text,
            "topic": it.get("topic"),
            "source_file": it.get("source_file") or name,
            "source_id": it.get("source_id"),
            "tier": it.get("tier") or tier,
            "vaccine_ids": vaccine_ids,
            "page": None,
        })
        n += 1
    print(f"{name}: {n} chunks")

CHUNKS_PATH.parent.mkdir(parents=True, exist_ok=True)
with CHUNKS_PATH.open("w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Wrote {len(rows)} → {CHUNKS_PATH}")
print("sample ids:", [r["chunk_id"] for r in rows[:8]])
assert any(r["chunk_id"].startswith("tcmr-") for r in rows), "Thiếu tcmr-*"
assert any("hdsd" in r["chunk_id"] for r in rows), "Thiếu *-hdsd-*"
print("OK — chạy cell tiếp theo")



chunks_tcmr_va_sang_loc.yaml: 18 chunks
chunks_hdsd_11_san_pham.yaml: 25 chunks
chunks_who_16_bai.yaml: 36 chunks
Wrote 79 → d:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\chunks\chunks.jsonl
sample ids: ['tcmr-001', 'tcmr-002', 'tcmr-003', 'tcmr-004', 'tcmr-005', 'tcmr-006', 'tcmr-007', 'tcmr-008']
OK — chạy cell tiếp theo


## 2) Load `chunks.jsonl`


In [5]:
chunks = []
with open(CHUNKS_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            chunks.append(json.loads(line))

print("n_chunks =", len(chunks))
texts = [(c.get("text") or "").strip() for c in chunks]
assert all(texts), "Có chunk text rỗng"
print("avg_len =", int(np.mean([len(t) for t in texts])))
print("sample keys", chunks[0].keys())
print("sample chunk_id", chunks[0].get("chunk_id"), "|", chunks[min(5, len(chunks)-1)].get("chunk_id"))


n_chunks = 79
avg_len = 213
sample keys dict_keys(['chunk_id', 'doc_id', 'title', 'text', 'topic', 'source_file', 'source_id', 'tier', 'vaccine_ids', 'page'])
sample chunk_id tcmr-001 | tcmr-006


## 3) Embed BGE-M3

Lần đầu tải model có thể mất vài phút.


In [6]:
from app.chatbot.rag.embedder import load_embedder

embedder = load_embedder(prefer_bge=True, corpus_for_tfidf=texts)
print("model =", embedder.model_name, "dim =", embedder.dim)

emb = embedder.encode(texts, batch_size=8)
print("embeddings shape", emb.shape, emb.dtype)
if "tfidf" in str(embedder.model_name).lower():
    print("CẢNH BÁO: đang TF-IDF fallback — production nên dùng BGE-M3")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model = BAAI/bge-m3 dim = 1024


Inference Embeddings: 100%|██████████| 10/10 [00:49<00:00,  4.96s/it]

embeddings shape (79, 1024) float32


## 4) Build & save FAISS


In [7]:
from app.chatbot.rag.vector_store import VectorStore

metas = []
for c in chunks:
    metas.append({
        "chunk_id": c.get("chunk_id"),
        "doc_id": c.get("doc_id"),
        "title": c.get("title"),
        "text": c.get("text"),
        "source_file": c.get("source_file"),
        "tier": c.get("tier"),
        "vaccine_ids": c.get("vaccine_ids") or [],
        "page": c.get("page"),
        "topic": c.get("topic"),
    })

store = VectorStore.build_from_embeddings(emb, metas, model_name=embedder.model_name)
print("index size", store.size, "dim", store.dim)

build_info = {
    "model": embedder.model_name,
    "dim": store.dim,
    "n_chunks": store.size,
    "built_at": datetime.now(timezone.utc).isoformat(),
    "chunks_path": str(CHUNKS_PATH),
    "sample_chunk_ids": [m["chunk_id"] for m in metas[:12]],
}
INDEX_DIR.mkdir(parents=True, exist_ok=True)
store.save(INDEX_PATH, META_PATH, build_info=build_info)
INFO_PATH.write_text(json.dumps(build_info, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved", INDEX_PATH)
print("saved", META_PATH)
print("saved", INFO_PATH)
print("sample_chunk_ids", build_info["sample_chunk_ids"])


index size 79 dim 1024
saved d:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\index\faiss.index
saved d:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\index\chunks_meta.json
saved d:\TTTN-VaxCare-2026\vaxcare-ai\knowledge\index\build_info.json
sample_chunk_ids ['tcmr-001', 'tcmr-002', 'tcmr-003', 'tcmr-004', 'tcmr-005', 'tcmr-006', 'tcmr-007', 'tcmr-008', 'tcmr-009', 'tcmr-010', 'scr-001', 'scr-002']


## 5) Smoke test retrieval


In [8]:
queries = [
    "Trẻ sơ sinh tiêm những mũi nào theo TCMR?",
    "Vaccine HPV phòng bệnh gì?",
    "Chống chỉ định tiêm Varivax?",
    "Khám sàng lọc trước tiêm chủng trẻ em",
]
for q in queries:
    qv = embedder.encode([q])[0]
    hits = store.search(qv, top_k=3, min_score=0.0)
    print("\nQ:", q)
    for h in hits:
        snippet = (h.get("text") or "")[:100].replace("\n", " ")
        cid = h.get("chunk_id") or h.get("doc_id") or ""
        print(f"  [{h['score']:.3f}] {cid} | {snippet}...")



Q: Trẻ sơ sinh tiêm những mũi nào theo TCMR?
  [0.621] tcmr-002 | Lịch cho trẻ sơ sinh theo TCMR: vắc xin viêm gan B đơn giá tiêm trong vòng 24 giờ sau sinh (liều sơ ...
  [0.610] tcmr-003 | Từ khi trẻ đủ 02 tháng tuổi, lịch TCMR gồm: vắc xin phối hợp có thành phần viêm gan B, vắc xin phối ...
  [0.582] tcmr-005 | Vắc xin viêm não Nhật Bản B theo TCMR tiêm 3 lần: lần 1 khi trẻ đủ 01 tuổi, lần 2 cách lần 1 từ 1 đế...

Q: Vaccine HPV phòng bệnh gì?
  [0.521] per-who-001 | Mục tiêu chính của tiêm ho gà là giảm ho gà nặng ở trẻ nhỏ. WHO khuyến cáo mọi trẻ, kể cả trẻ nhiễm ...
  [0.520] var-hdsd-001 | Varivax được chỉ định tiêm phòng thủy đậu cho người từ 12 tháng tuổi trở lên. Vaccine cũng dùng cho ...
  [0.511] tet-who-001 | Mục tiêu của WHO là bảo vệ suốt đời khỏi uốn ván bằng 6 liều: 3 liều cơ bản cộng 3 liều nhắc. Liều c...

Q: Chống chỉ định tiêm Varivax?
  [0.654] var-hdsd-004 | Không được dùng Varivax cho phụ nữ có thai và cần tránh mang thai trong 3 tháng sau khi tiêm. Chưa r...
 